# Fine-Tune Mistral-7B with LoRA in Colab

**A step-by-step guide to fine-tuning Mistral-7B using Low-Rank Adaptation (LoRA).**

 **No GPU?**
Use **Google Colab** (Runtime → Change runtime type → **T4 or A100 GPU**).

 **Estimated Time:**
~30–45 mins for 1 epoch on `databricks/databricks-dolly-15k`.

 **Storage:**
~15GB for the model + dataset (Colab provides **~80GB** for free).

---
## What You’ll Learn
1. Load and preprocess a dataset for instruction fine-tuning.
2. Apply **4-bit quantization** to fit Mistral-7B in limited GPU memory.
3. Fine-tune with **LoRA** (parameter-efficient, ~1% of original parameters).
4. Evaluate and generate responses with your custom model.

---
## 🔧 Setup
Click **Runtime → Run all** to execute the entire notebook.

---
## 1 Install Dependencies

In [ ]:
# Install required packages
!pip install -q \
    torch==2.1.2 \
    transformers==4.38.2 \
    peft==0.8.2 \
    accelerate==0.25.0 \
    bitsandbytes==0.41.3 \
    datasets==2.16.1 \
    sentencepiece \
    scipy \
    pandas \
    matplotlib \
    tqdm

# Restart the kernel to apply changes (Colab-specific)
import os
if 'COLAB_GPU' in os.environ:
    print(" Restarting kernel to apply installed packages...")
    os.kill(os.getpid(), 9)
else:
    print(" Dependencies installed.")


## 2 Imports and Setup

In [ ]:

import json
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    GenerationConfig,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔍 Using device: **{device}**")
if device == "cuda":
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


## 3 Load and Preprocess Dataset
We’ll use **`databricks/databricks-dolly-15k`**, a high-quality instruction-following dataset.

In [ ]:
# Load dataset
dataset_name = "databricks/databricks-dolly-15k"
print(f" Loading dataset: **{dataset_name}**...")
dataset = load_dataset(dataset_name, split="train")

# Show sample
print("\n👀 Sample from the dataset:")
print(json.dumps(dataset[0], indent=2))

# Load tokenizer
model_name = "mistralai/Mistral-7B-v0.1"
print(f" Loading tokenizer: **{model_name}**...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set pad token

# Preprocess function
def preprocess(example, max_length=512):
    instruction = example["instruction"]
    context = example.get("context", "")
    response = example["response"]

    # Format prompt (adjust based on your needs)
    prompt = f"""### Instruction:
{instruction}

### Context:
{context}

### Response:
{response}"""

    # Tokenize
    inputs = tokenizer(
        prompt,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    # For causal LM, labels = input_ids
    inputs["labels"] = inputs["input_ids"].clone()
    return inputs

# Apply preprocessing
print(" Preprocessing dataset...")
dataset = dataset.map(
    lambda x: preprocess(x, max_length=512),
    remove_columns=dataset.column_names,
    batched=False
)

# Filter out examples longer than max_length
dataset = dataset.filter(lambda x: len(x["input_ids"]) <= 512)
print(f" Dataset ready! **{len(dataset)}** examples after filtering.")


## 4 Load Mistral-7B with 4-bit Quantization
We use **`bitsandbytes`** to reduce memory usage.

In [ ]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f" Loading model: **{model_name}** with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for LoRA
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False  # Disable cache for training
print(" Model loaded and prepared for LoRA!")



## 5 Apply LoRA Configuration
LoRA (Low-Rank Adaptation) reduces trainable parameters by **~1000x** vs. full fine-tuning.

In [ ]:
# LoRA config
lora_config = LoraConfig(
    r=8,                          # Rank
    lora_alpha=16,               # Scaling factor
    lora_dropout=0.05,           # Dropout
    bias="none",                 # No bias
    task_type="CAUSAL_LM",       # Causal language modeling
    target_modules=[             # Target these layers
        "q_proj", "k_proj", "v_proj", "o_proj"
    ]
)

# Apply LoRA
print(" Applying LoRA...")
model = get_peft_model(model, lora_config)

# Print trainable parameters
def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f" Trainable parameters: **{trainable_params:,}**")
    print(f" All parameters: **{all_params:,}**")
    print(f" % Trainable: **{100 * trainable_params / all_params:.2f}%**")

print_trainable_parameters(model)


## 6 Training Setup
Configure hyperparameters and create the `Trainer`.


In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="results",           # Output directory
    per_device_train_batch_size=4, # Batch size per GPU
    gradient_accumulation_steps=4, # Accumulate gradients
    learning_rate=2e-5,            # Learning rate
    num_train_epochs=1,             # Epochs
    max_steps=100,                  # Limit steps for demo (remove for full training)
    logging_steps=10,               # Log every 10 steps
    save_steps=50,                 # Save every 50 steps
    save_total_limit=2,            # Max checkpoints
    report_to="none",              # Disable WandB/TensorBoard
    optim="paged_adamw_8bit",      # Optimizer
    lr_scheduler_type="cosine",    # LR scheduler
    warmup_steps=50,               # Warmup steps
    fp16=True,                     # Mixed precision
    bf16=False,
)

# Data collator
def data_collator(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
    }

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)



## 7 Train the Model!
 **Note:** For a full run, remove `max_steps=100` in `training_args` and increase `num_train_epochs` to **3**.

In [ ]:
print(" Starting training...")
train_result = trainer.train()
print(" Training complete!")

# Save metrics
metrics = train_result.metrics
print("\n Training Metrics:")
for k, v in metrics.items():
    print(f"- **{k}**: {v:.4f}")

# Plot loss
plt.figure(figsize=(10, 5))
plt.plot(trainer.state.log_history[1:], label="Train Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.grid()
plt.show()


## 8 Evaluate the Model
Compare **before** and **after** fine-tuning on a sample prompt.

In [ ]:
# Generate function
def generate_response(model, tokenizer, prompt, max_new_tokens=256, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Test prompt
prompt = """### Instruction:
Explain the concept of parameter-efficient fine-tuning (PEFT) in simple terms.

### Response:"""

# Generate with base model (if not quantized)
print(" **Base Model (Mistral-7B) Response:**")
try:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    base_response = generate_response(base_model, tokenizer, prompt, max_new_tokens=128)
    print(base_response)
except:
    print("(Base model not loaded due to memory constraints.)")

# Generate with fine-tuned model
print("\n **Fine-Tuned Model Response:**")
fine_tuned_response = generate_response(model, tokenizer, prompt, max_new_tokens=128)
print(fine_tuned_response)


## 9 Save the Model
Save the LoRA adapter and tokenizer for later use.

In [ ]:
# Save directory
save_dir = "results/llm-finetuned-lora"
print(f" Saving model to: **{save_dir}**...")

# Save LoRA adapter
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Save config
with open(f"{save_dir}/training_config.json", "w") as f:
    json.dump({
        "model": model_name,
        "dataset": dataset_name,
        "lora_config": lora_config.__dict__,
        "training_args": vars(training_args),
    }, f, indent=2)

print(" Model saved! Upload this to [Hugging Face Hub](https://huggingface.co/) to share it.")

##  Next Steps
### To Improve This Notebook:
1. **Train Longer**: Remove `max_steps=100` and set `num_train_epochs=3`.
2. **Use a Larger Dataset**: Try `HuggingFaceH4/ultrachat_200k`.
3. **Add QLoRA**: Combine 4-bit quantization + LoRA for even better efficiency.
4. **Evaluate on Benchmarks**: Use `lm-evaluation-harness` to test on **MMLU, TruthfulQA, etc.**

### To Use in Production:
**Deploy with FastAPI**:
   ```python
   from fastapi import FastAPI
   app = FastAPI()
   @app.post("/generate")
   def generate(text: str):
       return {"response": generate_response(model, tokenizer, text)}